# Test My Agent (Eigentiki)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tolani007/sft-coding-agent/blob/main/notebooks/inference.ipynb)

I use this notebook to test my new coding agent. I want to see if it can use tools correctly. I load the model and give it a task.

## 1. Install Tools

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes


## 2. Test the Agent
I load the model from Hugging Face. Then I ask it to write code and run it using a tool.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

model_id = "focustiki/eigentiki"
print(f"Loading model: {model_id}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 8192,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

print("\nModel loaded. Preparing the test task...")

messages = [
    {
        "role": "system", 
        "content": "You are a smart coding assistant. You have access to a tool named `run_python_code(code: str)`. You must use this tool if you need to run code."
    },
    {
        "role": "user", 
        "content": "Can you write a python script to calculate the square root of 144 and run it for me?"
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("\nAsking the agent to solve the task...")
outputs = model.generate(**inputs, max_new_tokens=250, use_cache=True, temperature=0.1)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- Full Output ---")
print(response)
print("----------------------")
